### Cena nieruchomości

#### Import bibliotek

In [ ]:
# Podstawowe bibliteki
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import PartialDependenceDisplay
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
import glob
import os

# Ustawienia
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
import warnings
warnings.filterwarnings("ignore")

#### Wczytywanie daych

##### Sprzedaż

In [ ]:
sales_files = sorted(glob.glob("apartments_pl_*.csv"))

print("Znalezione pliki sprzedaży:")
for f in sales_files:
    print(f)

sales_list = []

for file in sales_files:
    temp_df = pd.read_csv(file)
    parts = file.replace(".csv", "").split("_")
    year = parts[-2]
    month = parts[-1]
    temp_df["year"] = int(year)
    temp_df["month"] = int(month)
    temp_df["date"] = pd.to_datetime(temp_df[["year", "month"]].assign(day=1))
    sales_list.append(temp_df)

sales = pd.concat(sales_list, ignore_index=True)

print("Zakres dat:", sales["date"].min(), "do", sales["date"].max())
print("Liczba obserwacji:", sales.shape[0])

sales.head()

##### Wynajem

In [ ]:
rent_files = sorted(glob.glob("apartments_rent_pl_*.csv"))

print("Znalezione pliki najmu:")
for f in rent_files:
    print(f)

rent_list = []

for file in rent_files:
    temp_df = pd.read_csv(file)
    parts = file.replace(".csv", "").split("_")
    year = parts[-2]
    month = parts[-1] 
    temp_df["year"] = int(year)
    temp_df["month"] = int(month)
    temp_df["date"] = pd.to_datetime(temp_df[["year", "month"]].assign(day=1))
    rent_list.append(temp_df)

rent = pd.concat(rent_list, ignore_index=True)

print("Zakres dat:", rent["date"].min(), "do", rent["date"].max())
print("Liczba obserwacji:", rent.shape[0])

rent.head()

##### Makro

In [ ]:
macro = pd.read_csv("ceny 2023_2024.csv")
macro.info()
macro.head()

#### Filtrowanie miasta

In [ ]:
sales_waw = sales[sales["city"] == "warszawa"].copy()
rent_waw = rent[rent["city"] == "warszawa"].copy()

print("Sprzedaż - Warszawa:", sales_waw.shape)
print("Wynajem - Warszawa:", rent_waw.shape)

#### Przygotowanie i łączenie danych

In [ ]:
macro = macro.sort_values("date")
macro["inflation_mom"] = macro["inflation"].pct_change() * 100
macro["fuel_change"] = macro["fuel_price"].pct_change() * 100
macro["rate_change"] = macro["interest_rate"].diff()
macro = macro.dropna()

In [ ]:
macro["date"] = pd.to_datetime(macro["date"])

In [ ]:
sales_waw = sales_waw.merge(macro, on="date", how="left")
sales_waw.head()

#### Jakość danych

##### Sprzedaż

In [ ]:
print("Podstawowe informacje:")
sales_waw.info()

print("Braki w danych:")
missing = sales_waw.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0])

print("Statystyki opisowe:")
sales_waw.describe()

##### Wynajem

In [ ]:
print("Podstawowe informacje:")
rent_waw.info()

print("Braki w danych:")
missing_rent = rent_waw.isnull().sum().sort_values(ascending=False)
print(missing_rent[missing_rent > 0])

print("Statystyki opisowe:")
rent_waw.describe()

#### Cena za metr^2

In [ ]:
# Sprzedaż
sales_waw["price_per_m2"] = sales_waw["price"] / sales_waw["squareMeters"]
sales_waw["price_per_m2"].describe()

In [ ]:
# Wynajem
rent_waw["price_per_m2"] = rent_waw["price"] / rent_waw["squareMeters"]
rent_waw["price_per_m2"].describe()

#### Czyszczenie danych

In [ ]:
# Min wartość
sales_waw = sales_waw[sales_waw["squareMeters"] > 15]

# Czyszczenie
q_low = sales_waw["price_per_m2"].quantile(0.01)
q_high = sales_waw["price_per_m2"].quantile(0.99)

sales_waw = sales_waw[
    (sales_waw["price_per_m2"] > q_low) &
    (sales_waw["price_per_m2"] < q_high)
]

print("Po czyszczeniu:", sales_waw.shape)

In [ ]:
# Min wartość
rent_waw = rent_waw[rent_waw["squareMeters"] > 15]

# Czyszczenie
q_low = rent_waw["price_per_m2"].quantile(0.01)
q_high = rent_waw["price_per_m2"].quantile(0.99)

rent_waw = rent_waw[
    (rent_waw["price_per_m2"] > q_low) &
    (rent_waw["price_per_m2"] < q_high)
]

print("Po czyszczeniu:", rent_waw.shape)

#### Analiza trendów czasowych

##### Średnia cena za m^2 wg. miesięcy - Sprzedaż

In [ ]:
monthly_sales = sales_waw.groupby("date")["price_per_m2"].mean()

plt.figure(figsize=(10,6))
monthly_sales.plot(marker="o")
plt.title("Średnia cena za m2 – Warszawa (sprzedaż)")
plt.ylabel("Cena za m2")
plt.xlabel("Data")
plt.show()

monthly_sales

##### Tempo zmiany w analizowanym okresie - Sprzedaż

In [ ]:
growth_sales = (monthly_sales.iloc[-1] / monthly_sales.iloc[0] - 1) * 100
print("Zmiana % w okresie:", round(growth_sales, 2), "%")

##### Średnia cena za m^2 wg. miesięcy - Wynajem

In [ ]:
monthly_rent = rent_waw.groupby("date")["price_per_m2"].mean()

plt.figure(figsize=(10,6))
monthly_rent.plot(marker="o")
plt.title("Średnia cena wynajmu za m2 – Warszawa")
plt.ylabel("Cena najmu za m2")
plt.xlabel("Data")
plt.show()

##### Tempo zmiany w analizowanym okresie - Wynajem

In [ ]:
growth_rent = (monthly_rent.iloc[-1] / monthly_rent.iloc[0] - 1) * 100
print("Zmiana % w okresie (Wynajem):", round(growth_rent, 2), "%")

##### Stosunek czynszu do ceny

In [ ]:
common_dates = monthly_sales.index.intersection(monthly_rent.index)

rent_to_price = (monthly_rent[common_dates] * 12) / monthly_sales[common_dates]

plt.figure(figsize=(10,6))
rent_to_price.plot(marker="o")
plt.title("Stosunek ceny wynajem/sprzedaż (roczna stopa brutto)")
plt.ylabel("Relacja czynsz/cena")
plt.xlabel("Data")
plt.show()

rent_to_price

#### Prosty model estymacji ceny w następnym miesiącu

In [ ]:
monthly_df = monthly_sales.reset_index()
monthly_df["t"] = np.arange(len(monthly_df))

X_time = monthly_df[["t"]]
y_time = monthly_df["price_per_m2"]

trend_model = LinearRegression()
trend_model.fit(X_time, y_time)

# prognoza na kolejny miesiąc
next_t = np.array([[len(monthly_df)]])
forecast = trend_model.predict(next_t)

print("Prognozowana średnia cena za m2 w kolejnym miesiącu:",
      round(forecast[0], 2))

#### Predykcja cen

##### Model hedoniczny

###### Budowa modelu hedonicznego

In [ ]:
df = sales_waw.copy()

# Log ceny
df["log_price_m2"] = np.log(df["price_per_m2"])

features = [
    "squareMeters",
    "rooms",
    "floor",
    "floorCount",
    "buildYear",
    "centreDistance",
    "poiCount"    
]

df_model = df[features + ["log_price_m2", "date"]].dropna()

# Miesiąc jako string
df_model["month"] = df_model["date"].dt.to_period("M").astype(str)

# Dummy months
month_dummies = pd.get_dummies(df_model["month"], drop_first=True).astype(float)

# Skalowanie zmiennych ciągłych
scaler = StandardScaler()
X_scaled_numeric = scaler.fit_transform(df_model[features])
X_scaled = pd.DataFrame(X_scaled_numeric, columns=features, index=df_model.index)

# Łączenie
X = pd.concat([X_scaled, month_dummies], axis=1)

# Dodanie stałej
X = sm.add_constant(X).astype(float)
y = df_model["log_price_m2"].astype(float)

# Estymacja
model = sm.OLS(y, X).fit()
print(model.summary())

###### Budowa indeksu cen

In [ ]:
# Współczynniki miesięczne
month_coefs = model.params.filter(like="202")

# Miesiąc bazowy
base_month = df_model["month"].min()

# DF indeksu
index_df = pd.DataFrame({
    "month": [base_month] + list(month_coefs.index),
    "beta": [0] + list(month_coefs.values)
})

# Konwersje i sortowanie
index_df["index"] = 100 * np.exp(index_df["beta"])
index_df["month"] = pd.to_datetime(index_df["month"])
index_df = index_df.sort_values("month")

plt.figure(figsize=(10,6))
plt.plot(index_df["month"], index_df["index"], marker="o")
plt.title("Hedoniczny indeks cen mieszkań – Warszawa")
plt.ylabel("Indeks (100 = miesiąc bazowy)")
plt.xlabel("Data")
plt.show()

index_df

###### Dynamika indeksu

In [ ]:
index_df["mom_growth_%"] = index_df["index"].pct_change() * 100
index_df

###### Przeliczenie danch na cały okres

In [ ]:
cumulative_growth = (index_df["index"].iloc[-1] / index_df["index"].iloc[0] - 1) * 100
print("Łączny wzrost:", round(cumulative_growth,2), "%")

###### Predykcja ARiMA

In [ ]:
# Predykcja
ts = index_df.set_index("month")["index"]
ts_log = np.log(ts)
model_arima = ARIMA(ts_log, order=(1,1,0))
fit = model_arima.fit()
forecast = fit.get_forecast(steps=3)

# powrót do indeksu
forecast_index = np.exp(forecast.predicted_mean)
print(fit.summary())
print(forecast_index)

###### Przediały ufności

In [ ]:
ci = forecast.conf_int()
ci_exp = np.exp(ci)

plt.figure(figsize=(10,6))
plt.plot(ts.index, ts, label="Historyczne")
plt.plot(forecast_index.index, forecast_index, label="Prognoza")
plt.fill_between(
    forecast_index.index,
    ci_exp.iloc[:,0],
    ci_exp.iloc[:,1],
    alpha=0.3
)
plt.legend()
plt.title("Prognoza hedonicznego indeksu cen")
plt.show()

##### Model makroekonomiczny

###### Budowa modelu makro

In [ ]:
df = sales_waw.copy()

# Log ceny
df["log_price_m2"] = np.log(df["price_per_m2"])

# Opóźnienia
df["inflation_lag1"] = df["inflation_mom"].shift(1)
df["inflation_lag3"] = df["inflation_mom"].shift(3)
df["rate_lag3"] = df["rate_change"].shift(3)
df["price_lag1"] = df["log_price_m2"].shift(1)

# Features
macro_features = [
    "inflation_lag1",
    "inflation_lag3",
    "rate_change",
    "rate_lag3",
    "fuel_change",
    "price_lag1"
]

# Datast
df_model_macro = df[macro_features + ["log_price_m2"]].dropna()

# Skalowanie
scaler_macro = StandardScaler()
X_scaled = scaler_macro.fit_transform(df_model_macro[macro_features])
X = pd.DataFrame(X_scaled, columns=macro_features, index=df_model_macro.index)
X = sm.add_constant(X)

y = df_model_macro["log_price_m2"]

# Estymacja
model_macro = sm.OLS(y, X).fit()
print(model_macro.summary())

###### Scenariusze makro

In [ ]:
# Daty prognozy
future_dates = forecast_index.index

# Scenariusz bazowy
macro_base = pd.DataFrame({
    "inflation_mom": [0.2, 0.3, 0.2],
    "rate_change": [-0.25, -0.25, 0],
    "fuel_change": [0.5, 0.2, 0.1]
})

###### Funkcja predykcji makro

In [ ]:
def predict_macro_impact(df_macro):
    df_macro = df_macro[macro_features]
    df_scaled = scaler_macro.transform(df_macro)
    df_scaled = pd.DataFrame(df_scaled, columns=macro_features)
    df_scaled = sm.add_constant(df_scaled, has_constant='add')
    pred_log = model_macro.predict(df_scaled)
    return pred_log.diff().fillna(0)

In [ ]:
def build_macro_with_lags(df_hist, df_future):
    cols = ["inflation_mom", "rate_change", "fuel_change"]
    df_all = pd.concat([df_hist[cols], df_future], ignore_index=True)
    df_all["inflation_lag1"] = df_all["inflation_mom"].shift(1)
    df_all["inflation_lag3"] = df_all["inflation_mom"].shift(3)
    df_all["rate_lag3"] = df_all["rate_change"].shift(3)
    df_all["price_lag1"] = df_hist["log_price_m2"].iloc[-1]
    df_future_final = df_all.iloc[-len(df_future):].copy()
    return df_future_final[macro_features]

df_hist = df.tail(6)
macro_base_lagged = build_macro_with_lags(df_hist, macro_base)

In [ ]:
base_path = forecast_index

def build_macro_index(df_macro, strength=1.0):
    impact = predict_macro_impact(df_macro)
    impact = impact * strength
    impact = impact.clip(-0.005, 0.005)
    impact = impact.rolling(2, min_periods=1).mean()
    deviation = [1]

    for i in impact:
        deviation.append(deviation[-1] * np.exp(i))

    deviation = pd.Series(deviation[1:], index=future_dates)
    return forecast_index * deviation

macro_base_index = build_macro_index(macro_base_lagged, 1.0)

###### Wykres wpływu zmiennych makro 

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(ts.index, ts, label="Indeks historyczny", marker="o")
plt.plot(forecast_index.index, forecast_index, linestyle="--", label="Prognoza ARIMA")
plt.plot(future_dates, macro_base_index, linestyle=":", label="Makro - scenariusz bazowy")
plt.legend()
plt.title("Prognoza cen mieszkań: ARIMA vs scenariusze makro")
plt.xlabel("Data")
plt.ylabel("Indeks cen")
plt.grid()
plt.show()

###### Indeks cen makro

In [ ]:
df_results = pd.DataFrame({"data": list(ts.index) + list(future_dates)})
df_results = df_results.set_index("data")

df_results["historyczny_indeks"] = ts
df_results["prognoza_ARIMA"] = forecast_index
df_results["makro_base"] = macro_base_index
df_results = df_results.round(2)

df_results.tail(10)

##### Model oceny determinantów ceny

###### RandomForest

In [ ]:
features = [
    "squareMeters",
    "rooms",
    "floor",
    "floorCount",
    "buildYear",
    "centreDistance",
    "poiCount"
]

df_rf = sales_waw.copy()
df_rf = df_rf[features + ["price_per_m2"]].dropna()

X = df_rf[features]
y = df_rf["price_per_m2"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2:", r2)

###### Ważność zmiennych

In [ ]:
importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importance)

###### Hierarchia zmiennych

In [ ]:
importance.plot(kind="bar")
plt.title("Hierarchia zmiennych - Random Forest")
plt.ylabel("Znaczenie")
plt.xlabel("Zmienne")
plt.grid()
plt.show()

###### Zależność cech

In [ ]:
PartialDependenceDisplay.from_estimator(rf, X, ["centreDistance", "squareMeters"])
plt.show()

###### Porrówanie mdeli

In [ ]:
features = [
    "buildYear",
    "centreDistance",
    "floor",
    "floorCount",
    "poiCount",
    "rooms",
    "squareMeters"
]

ols_clean = model.params[features].abs()
rf_clean = importance[features]

comparison = pd.DataFrame({"OLS": ols_clean, "RandomForest": rf_clean})
comparison = comparison.sort_values(by="RandomForest", ascending=False)
comparison

#### Walidacja

##### Błędy predykcji

In [ ]:
# predykcja
y_pred = model.predict(X)

# powrót z log
y_true = np.exp(y)
y_pred_price = np.exp(y_pred)

# błędy
mae = mean_absolute_error(y_true, y_pred_price)
rmse = np.sqrt(mean_squared_error(y_true, y_pred_price))

print("MAE:", round(mae,2))
print("RMSE:", round(rmse,2))

##### Diagnostyka reszt modelu

In [ ]:
residuals = model.resid

plt.figure()
sns.histplot(residuals, kde=True)
plt.title("Rozkład reszt modelu")
plt.show()

plt.figure()
plt.scatter(y_pred, residuals)
plt.axhline(0)
plt.title("Reszty vs wartości dopasowane")
plt.show()

##### Backtest - ARiMA

In [ ]:
train = ts_log[:-2]
test = ts_log[-2:]

model_test = ARIMA(train, order=(1,1,1)).fit()
forecast_test = model_test.forecast(steps=2)
forecast_test = np.exp(forecast_test)
test_real = np.exp(test)

print("Rzeczywiste:")
print(test_real)

print("Prognoza:")
print(forecast_test)

##### Testy stabilności

In [ ]:
plt.figure()
plt.scatter(y_true, y_pred_price)
plt.plot([y_true.min(), y_true.max()],
         [y_true.min(), y_true.max()])
plt.title("Rzeczywiste vs przewidywane ceny")
plt.xlabel("Rzeczywiste")
plt.ylabel("Przewidywane")
plt.show()